#  **ICT303 - Assignment 2**

**Your name:**

**Student ID: **

**Email: **

In this assignment, you will build and train a deep learning model for solving a problem of your choice.


You are required to:
- Think of a practical problem that you would like to solve. The problem can be related, but not limted to, object detection and recognition from images, text analysis, speech analysis, image unpainting, converting images to artistic painting, action recognition (from images or videos), image to text (i.e., generating textual description for images or videos), or texrt to image (generating images from text) etc.,
- Find an appropriate data set to train and test the model you will develop. Note that the dataset should contain enough data (with groundtruth labels) so that when used for training, the model can generalize well to unseen data.
- Design a neural network that will solve the problem
- Train the neural network on your training data and then evaluate its performance on test data
- Analyze the performance of the network you developed and discuss its limitations.

**What to submit:**
- A colab notebook that describes:
 - The problem you would like to solve **[10 Marks]**
 - The dataset that you will use to train and test the deep learning model that you will develop **[10 marks]**
 - A diagram that describes the architecture of the neural network that you developed **[10 marks]**
 - Performance curves - this is includes the loss curves as well as the accuracy **[10 marks]**
 - A discussion, analysis and justification of the different choices you made and their effect on the performance **[15 marks]**
 - A discussion, analysis of the limitations of your method. You can also show failure cases and try to understand why did it fail on these cases **[15 marks]**

- Source code that runs - this includes both code for training and testing **[30 marks]**

You also need to submit the dataset you used for training and testing, or alternatively provide the code that downloads the data.

Make sure you reference all sources from which you took information.

You are allowed to use existing neural networks (not required to implement them from scratch). But, you must customize the architecture to the problem you want to solve.

**Where to find datasets?**
- [Kaggle competition](https://www.kaggle.com/c/dog-breed-identification) is a good source.
- You can also look at this [wikipedia site[(https://en.wikipedia.org/wiki/List_of_datasets_for_machine-learning_research)

If you are thinking of a specific problem and were unable to find a suitable dataset, please talk to me during the lecture or lab and we will search together.

**Recommended timeline**
- Week 1 of Assignement release: identify 2 or 3 problems of interest, a find dataset for each of the problem and try to understand how to load the data and how it is organised. Discuss it with the UC during the lab session or via email.
- Week 2:
 - Make sure your dataloader works proply and you are able to load the data and structure it in a way that neural networks can use them.
 - Design your network architecture
- Week 3: Network architecture implemented and training and testing done. Evaluate the performance
- Week 4: Finetune the network architecture and the hyperparameters to improve the performance. Write the report for submission.

It is highly recommended that you follow this timeline. The earlier you start training and testing, the more time you will have to finetune your solution and achieve a better performance.


# Movie Recommendation using Neural Collaboration Filtering Model

dataset - https://www.kaggle.com/datasets/samlearner/letterboxd-movie-ratings-data?

In [1]:
print("tensorboard extension loaded")

tensorboard extension loaded


## import and dependencies

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import tqdm
import os
import shutil
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import mean_squared_error
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import kagglehub

# Download latest version
path = kagglehub.dataset_download("samlearner/letterboxd-movie-ratings-data")

print("Path to dataset files:", path)


# Load CSV files



Path to dataset files: /home/kim/.cache/kagglehub/datasets/samlearner/letterboxd-movie-ratings-data/versions/6


In [3]:
#filtered = merged_data_df[merged_data_df['movie_title'].str.contains('lord of the rings', case=False, na=False)]
#print(filtered[['movie_title', 'popularity']])

## Neural Collboration Filtering Model Architecture



## NCF Model Class

In [4]:
class NCF(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=8, lr=1e-3, optimizer_type="adam"):
        super(NCF, self).__init__()
        self.lr = lr
        self.optimizer_type = optimizer_type

        # Embeddings for GMF
        self.user_embedding_gmf = nn.Embedding(num_users, embedding_dim)
        self.item_embedding_gmf = nn.Embedding(num_items, embedding_dim)

        # Embeddings for MLP
        self.user_embedding_mlp = nn.Embedding(num_users, embedding_dim)
        self.item_embedding_mlp = nn.Embedding(num_items, embedding_dim)

        # MLP Layers
        self.mlp_layers = nn.Sequential(
            nn.Linear(embedding_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )

        # Final prediction layer
        self.output_layer = nn.Linear(embedding_dim + 16, 1)  # GMF (8) + MLP output (16)

    def forward(self, user_indices, item_indices):
        # GMF embedded
        gmf_user = self.user_embedding_gmf(user_indices)
        gmf_item = self.item_embedding_gmf(item_indices)
        # GMF output
        gmf_output = gmf_user * gmf_item  # element-wise product

        # MLP embedded
        mlp_user = self.user_embedding_mlp(user_indices)
        mlp_item = self.item_embedding_mlp(item_indices)

        # MLP input
        mlp_input = torch.cat([mlp_user, mlp_item], dim=-1)

        #MLP output
        mlp_output = self.mlp_layers(mlp_input)

        # NeuMF Layer (GMF x MLP)
        combined = torch.cat([gmf_output, mlp_output], dim=-1)

        # Final prediction
        prediction = self.output_layer(combined).squeeze(-1)

        return prediction

    def loss(self, y_hat, y):
        return nn.MSELoss()(y_hat, y)

    def configure_optimizers(self):
        if self.optimizer_type == "adam":
            return optim.Adam(self.parameters(), lr=self.lr)
        elif self.optimizer_type == "sgd":
            return optim.SGD(self.parameters(), lr=self.lr)
        else:
            return optim.Adam(self.parameters(), lr=self.lr)

## Trainer Class

In [5]:
class Trainer:

  def __init__(self, tb, n_epochs = 3):
    self.max_epochs = n_epochs
    self.writer = tb  # the tensorboard instance
    return

  def fit(self, model, data, validation_data):
    self.data = data
    self.validation_data = validation_data

    # configure the optimizer
    self.optimizer = model.configure_optimizers()
    #self.scheduler = StepLR(self.optimizer, step_size=5, gamma=0.1)
    self.model     = model

    for epoch in range(self.max_epochs):
      print(f"\nEpoch {epoch + 1}/{self.max_epochs}")
      self.fit_epoch()
      self.validate_epoch()
      #self.scheduler.step()
      # Logging the average training loss so that it can be visualized in the tensorboard
      self.writer.add_scalar("Training Loss", self.avg_training_loss, epoch)
      self.writer.add_scalar("Validation Loss", self.avg_val_loss, epoch)

    print("Training process has finished")

  def fit_epoch(self):

    self.model.train()
    current_loss = 0.0
    self.avg_training_loss = 0.0

    # iterate over the DataLoader for training data
    for i, data in enumerate(tqdm(self.data, desc="Training")):
      # Get input
      (inputs, target) = data
      user, item = inputs  # instead of: inputs, target
      user, item, target = user.to(device), item.to(device), target.to(device)


      # Clear gradient buffers because we don't want any gradient from previous
      # epoch to carry forward, dont want to cummulate gradients
      self.optimizer.zero_grad()

      # get output from the model, given the inputs
      outputs = self.model(user, item)

      # get loss for the predicted output
      loss = self.model.loss(outputs, target)

      # get gradients w.r.t to the parameters of the model
      loss.backward()

      # update the parameters (perform optimization)
      self.optimizer.step()

      # Let's print some statistics (average of the training loss over minibatches of 500 data items)
      current_loss += loss.item()

      # Adding training loss
      self.avg_training_loss += loss.item()

      if i % 500 == 499:
          print('Loss after mini-batch %5d: %.3f' %
                (i + 1, current_loss / 500))
          current_loss = 0.0

    # The average training loss
    self.avg_training_loss = self.avg_training_loss / i # to get the average
    print(f"Training Loss (avg): {self.avg_training_loss:.4f}")

  def validate_epoch(self):

    self.model.eval()
    total_loss = 0.0
    self.avg_val_loss = 0.0

    with torch.no_grad():
      # iterate over the DataLoader for training data
      for i, data in enumerate(self.validation_data):
        # Get input
        (inputs, target) = data
        user, item = inputs  # instead of: inputs, target
        user, item, target = user.to(device), item.to(device), target.to(device)

        # get output from the model, given the inputs
        outputs = self.model(user, item)

        # get loss for the predicted output
        loss = self.model.loss(outputs, target)

        total_loss += loss.item()

      # The average training loss
      self.avg_val_loss = total_loss / (i + 1) # to get the   average
      print(f"Validation Loss (avg): {self.avg_val_loss:.4f}")

In [6]:
class Dataloader:
    def __init__(self, dataframe, batch_size=1024, validation_split=0.2):
        self.data = dataframe.copy()
        self.data["label"] = self.data["rating_val"] / 2.0  # Normalized to 0–2.5

        # Map user and item IDs to indices
        self.user2idx = {u: i for i, u in enumerate(self.data["user_id"].unique())}
        self.item2idx = {m: i for i, m in enumerate(self.data["movie_id"].unique())}

        self.data["user"] = self.data["user_id"].map(self.user2idx)
        self.data["item"] = self.data["movie_id"].map(self.item2idx)

        self.num_users = len(self.user2idx)
        self.num_items = len(self.item2idx)
        self.data = self.data[["user", "item", "label"]]

        from sklearn.model_selection import train_test_split
        self.train, self.val = train_test_split(self.data, test_size=validation_split, random_state=42)

    def get_dataloaders(self):
        return self.train, self.val



class NCFDataset(Dataset):
    def __init__(self, data):
        self.data = data.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        user = torch.tensor(row["user"], dtype=torch.long)
        item = torch.tensor(row["item"], dtype=torch.long)
        label = torch.tensor(row["label"], dtype=torch.float)
        return (user, item), label


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

#Hyperparameters
embedding_dim=32
lr=0.001
batch_size=8192
n_epochs=10


Using device: cuda


In [8]:
movie_df = pd.read_csv(os.path.join(path,'movie_data.csv'), engine='python')
ratings_df = pd.read_csv(os.path.join(path,'ratings_export.csv'), engine='python')
merged_data_df = pd.merge(ratings_df, movie_df, on='movie_id')
filtered_reviews = merged_data_df[merged_data_df['popularity'] > 50]


In [9]:
dl = Dataloader(filtered_reviews, batch_size=batch_size, validation_split=0.2)

train_dataset,val_dataset = dl.get_dataloaders()

train_dataset = NCFDataset(train_dataset)
val_dataset = NCFDataset(val_dataset)


train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size,
                          num_workers=4, pin_memory=True)

print ("Done")

Done


In [10]:
model = NCF(num_users=dl.num_users, num_items=dl.num_items, embedding_dim=embedding_dim, lr=lr)
model.to(device)


if os.path.exists('runs/ncf_model'):
  shutil.rmtree('runs/ncf_model')
writer = SummaryWriter(log_dir="runs/ncf_model")

trainer = Trainer(tb=writer, n_epochs=n_epochs)
trainer.fit(model, train_loader, val_loader)





Epoch 1/10


Training: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 124/124 [00:22<00:00,  5.55it/s]

Training Loss (avg): 3.3057


Validation Loss (avg): 1.0486

Epoch 2/10


Training: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 124/124 [00:22<00:00,  5.51it/s]

...

Training Loss (avg): 0.5854


Validation Loss (avg): 0.5878

Epoch 10/10


Training: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 124/124 [00:22<00:00,  5.55it/s]

Training Loss (avg): 0.5773


Validation Loss (avg): 0.5826
Training process has finished


In [11]:
import numpy as np
def compute_rmse(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for (inputs, labels) in dataloader:
            user, item = inputs
            user, item = user.to(device), item.to(device)
            labels = labels.to(device)

            preds = model(user, item)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    mse = mean_squared_error(all_labels, all_preds)
    rmse = np.sqrt(mse)
    return rmse



rmse = compute_rmse(model, val_loader)
print(f"Test RMSE: {rmse:.4f}")



Test RMSE: 0.7633


In [29]:
def recommend_top_k_movies(
    model,
    dl: Dataloader,
    user_name: str,
    full_ratings_df: pd.DataFrame,
    k: int = 10
):

    model.eval()
    user_ratings = full_ratings_df[full_ratings_df["user_id"] == user_name]
    if user_ratings.empty:
        print(f"User '{user_name}' not found or has no ratings.")
        return

    top_rated = user_ratings.sort_values(by="rating_val", ascending=False).head(k)
    print(f"\nTop {k} movies rated by '{user_name}':")
    for i, row in enumerate(top_rated.itertuples(index=False), 1):
        print(f"{i}. {row.movie_title} — Rating: {row.rating_val}")

    # 1. Check if the user exists in the dataset
    if user_name not in dl.user2idx:
        print(f"User '{user_name}' not found in dataset.")
        return

    user_idx = dl.user2idx[user_name]

    # 2. Get all movie_ids rated by this user
    rated_movie_ids = set(
        full_ratings_df[full_ratings_df["user_id"] == user_name]["movie_id"]
    )

    # 3. Get all unseen movie_ids
    unseen_movies = [
        (item_idx, movie_id)
        for movie_id, item_idx in dl.item2idx.items()
        if movie_id not in rated_movie_ids
    ]

    if not unseen_movies:
        print("All movies have been rated by this user. No recommendations available.")
        return

    # 4. Predict ratings for unseen movies
    user_tensor = torch.tensor([user_idx] * len(unseen_movies), dtype=torch.long).to(device)
    item_tensor = torch.tensor([item_idx for item_idx, _ in unseen_movies], dtype=torch.long).to(device)

    with torch.no_grad():
        predictions = model(user_tensor, item_tensor)

    # 5. Select top-k movie_ids based on predicted ratings
    top_k_indices = torch.topk(predictions, k).indices.cpu().numpy()
    top_k_movie_ids = [unseen_movies[i][1] for i in top_k_indices]

    # 6. Get movie titles from the full_ratings_df
    movie_dict = dict(zip(full_ratings_df["movie_id"], full_ratings_df["movie_title"]))

    # 7. Display top-k recommendations
    print(f"\nTop {k} Movie Recommendations for '{user_name}':")
    for rank, movie_id in enumerate(top_k_movie_ids, 1):
        title = movie_dict.get(movie_id, "Unknown Title")
        print(f"{rank}. {title}")


In [34]:
recommend_top_k_movies(
    model=model,
    dl=dl,
    user_name="houndy",
    full_ratings_df=filtered_reviews,
    k=10
)



Top 10 movies rated by 'houndy':
1. A Silent Voice — Rating: 9
2. Drive — Rating: 8
3. Star Wars: The Last Jedi — Rating: 8

Top 10 Movie Recommendations for 'houndy':
1. The Shining
2. The Godfather
3. Parasite
4. Inglourious Basterds
5. Spirited Away
6. Spider-Man: Into the Spider-Verse
7. The Shawshank Redemption
8. The Iron Giant
9. The Lord of the Rings: The Return of the King
10. The Incredibles


In [23]:
%tensorboard --logdir runs

UsageError: Line magic function `%tensorboard` not found.


In [ ]:
#Observations of NCF Model: Optimizer=Adam, Epoch=20, lr=.0001
#2 hours to train
#Overfitting at Epoch 12
#Results of deathproof: very niche recommendations but to be fair, he has 2.3k reviews (no life)
#Results of ericanders: a few similar recommendations to deathproof, will look into this, probably something wrong with the model.
#Found out ericanders also have no life (1.5k reviews)
#Results of houndy: user with only 55 reviews. Most of his/her top movies are animation films, recommendations are niche but very different from the previous users

#users with many reviews = similar recommendations. I believe this is due to people watching the same popular movies
#a lot of the recommendations are non-english movies and very not well known

#Does not recommend relevant items for users with few ratings



In [ ]:
#Observations of NCF Model: Optimizer=Adam, Epoch=10, lr=.0001
#3.3min to Train
#No overfitting
#Important: I merged ratings_export and movie_data into one dataframe. I cleaned the dataframe, removing all unpopular movies (popularity = < 50, Lord of the rings is 80-90 in comparison).
#Results of deathproof: Much better. The model recommended popular media which he haven't watched yet. He/she likes sci-fi and adventure, most of the recommendation for him/her are sci-fi/Fiction and adventure. Based on his recommended movies, he/she likes artsy
#Results of houndy: Also better. From 55 movies, he/she only has 3 movies that wasnt wiped from the data cleaning. His/her top movie is an animation film, and most of his/her recommendations were also animation. For only 3 reviews, his/her recommendation looked great.

In [ ]:
#Limitation
#My current model is a basic Neural Collaboration Filter, it is strictly collabarative filtering and does not filter content (users), it only recommends movies based on other users #likeness. It does not have the ability to personalize according to the user's genre/director/studio bias (content-based filtering).